In [ ]:
import gymnasium as gym
import numpy as np

In [ ]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8


In [ ]:
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------

def policy_evaluation(env, policy, gamma=0.99, theta=1e-8):
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    V = np.zeros(n_states)

    while True:
        delta = 0

        for s in range(n_states):
            v = V[s]
            new_value = 0

            for a in range(n_actions):
                action_prob = policy[s, a]

                for prob, next_state, reward, done in env.P[s][a]:
                    new_value += action_prob * prob * (
                        reward + gamma * V[next_state] * (not done)
                    )

            V[s] = new_value
            delta = max(delta, abs(v - V[s]))

        if delta < theta:
            break

    return V

In [ ]:
# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------

def policy_improvement(env, V, gamma=0.99):
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    new_policy = np.zeros((n_states, n_actions))

    for s in range(n_states):
        action_values = np.zeros(n_actions)

        for a in range(n_actions):
            for prob, next_state, reward, done in env.P[s][a]:
                action_values[a] += prob * (
                    reward + gamma * V[next_state] * (not done)
                )

        best_action = np.argmax(action_values)
        new_policy[s, best_action] = 1.0

    return new_policy

In [ ]:
# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

def policy_iteration(env, gamma=0.99, theta=1e-8):
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    policy = np.zeros((n_states, n_actions))

    for s in range(n_states):
        random_action = np.random.randint(n_actions)
        policy[s, random_action] = 1.0

    iterations = 0

    while True:
        iterations += 1

        V = policy_evaluation(env, policy, gamma, theta)

        new_policy = policy_improvement(env, V, gamma)

        if np.array_equal(policy, new_policy):
            break

        policy = new_policy

    print("Total policy iterations:", iterations)

    return policy, V


In [ ]:

# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)


    print("\nOptimal Policy:")
    print(policy_grid)



In [ ]:
# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name: S LALIT CHANDRAN     ")
print("Register Number: 212223240077      ")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()

Total policy iterations: 7
Name: S LALIT CHANDRAN     
Register Number: 212223240077      

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

policy = np.zeros((n_states, n_actions))

for s in range(n_states):
    policy[s, np.random.randint(n_actions)] = 1.0

V_evaluation = policy_evaluation(env, policy)

print("Value Function before Policy improvement:")
print(np.round(V_evaluation.reshape(4, 4), 3))

optimal_policy, V_optimal = policy_iteration(env)

print("\nValue Function after Policy improvement &  Iteration:")
print(np.round(V_optimal.reshape(4, 4), 3))

Value Function before Policy improvement:
[[0.019 0.005 0.011 0.005]
 [0.033 0.    0.016 0.   ]
 [0.067 0.102 0.039 0.   ]
 [0.    0.204 0.517 0.   ]]
Total policy iterations: 7

Value Function after Policy improvement &  Iteration:
[[0.542 0.499 0.471 0.457]
 [0.558 0.    0.358 0.   ]
 [0.592 0.643 0.615 0.   ]
 [0.    0.742 0.863 0.   ]]
